In [4]:
import torch
import tiktoken
from importlib.metadata import version

In [5]:
print("PyTorch version:", torch.__version__)
print("tiktoken version:", version("tiktoken"))

PyTorch version: 2.5.1
tiktoken version: 0.9.0


In [6]:
tokenizer = tiktoken.get_encoding("gpt2")
print("Tokenizer name:", tokenizer.name)

Tokenizer name: gpt2


In [7]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Encoded integers:", integers)
print("Decoded text:", tokenizer.decode(integers))

Encoded integers: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Decoded text: Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [8]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [9]:
enc_sample = enc_text[:50]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print("x:", x)
print("y:    ", y)

x: [40, 367, 2885, 1464]
y:     [367, 2885, 1464, 1807]


In [10]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    target = enc_sample[i]
    print(f"{context} ---> {target}")

[40] ---> 367
[40, 367] ---> 2885
[40, 367, 2885] ---> 1464
[40, 367, 2885, 1464] ---> 1807


In [11]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    target = enc_sample[i]
    print(tokenizer.decode(context), "--->", tokenizer.decode([target]))

I --->  H
I H ---> AD
I HAD --->  always
I HAD always --->  thought


# Dataloader

In [12]:
from torch.utils.data import Dataset, DataLoader

In [93]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Tokenizer output is shorter than max_length"
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [94]:
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    number_workers=0,
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=number_workers,
    )
    return dataloader

In [95]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

### Test Dataloader

In [96]:
dataLoader = create_dataloader_v1(
    raw_text,
    batch_size=4,
    max_length=8,
    stride=4,
    shuffle=False,
    drop_last=True,
    number_workers=0,
)

In [98]:
data_iter = iter(dataLoader)
first_batch = next(data_iter)
print("First batch input IDs:", first_batch[0])
print("First batch target IDs:", first_batch[1])

First batch input IDs: tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [10899,  2138,   257,  7026, 15632,   438,  2016,   257],
        [15632,   438,  2016,   257,   922,  5891,  1576,   438]])
First batch target IDs: tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899],
        [ 3619,   402,   271, 10899,  2138,   257,  7026, 15632],
        [ 2138,   257,  7026, 15632,   438,  2016,   257,   922],
        [  438,  2016,   257,   922,  5891,  1576,   438,   568]])


## Create Token Embedding 

In [99]:
input_ids = torch.tensor([5,1,2,3,4])

In [100]:
vocob_size = 6
output_dim = 4

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocob_size, output_dim)
print(f"Embedding layer weights before training:\n {embedding_layer.weight.data}")

Embedding layer weights before training:
 tensor([[ 0.3374, -0.1778, -0.3035, -0.5880],
        [ 0.3486,  0.6603, -0.2196, -0.3792],
        [-0.1606, -0.4015,  0.6957, -1.8061],
        [ 1.8960, -0.1750,  1.3689, -1.6033],
        [-0.7849, -1.4096, -0.4076,  0.7953],
        [ 0.9985,  0.2212,  1.8319, -0.3378]])


In [101]:
print(embedding_layer(input_ids))

tensor([[ 0.9985,  0.2212,  1.8319, -0.3378],
        [ 0.3486,  0.6603, -0.2196, -0.3792],
        [-0.1606, -0.4015,  0.6957, -1.8061],
        [ 1.8960, -0.1750,  1.3689, -1.6033],
        [-0.7849, -1.4096, -0.4076,  0.7953]], grad_fn=<EmbeddingBackward0>)


## Position embedding

In [102]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [140]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=max_length,
    stride=max_length,
    shuffle=False,
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)


In [143]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [148]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [144]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
#print("Positional embedding layer weights before training:\n", pos_embedding_layer.weight.data)

In [145]:
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("Positional embeddings shape:", pos_embeddings.shape)
#print("Positional embeddings:", pos_embeddings)

Positional embeddings shape: torch.Size([4, 256])


In [151]:
input_embedding = token_embeddings + pos_embeddings
print(input_embedding.shape)

torch.Size([8, 4, 256])
